# 20 · Model Comparison

Notebook agregador para comparar LGWR/GWR, SAR, RF + Kriging y GNN bajo un protocolo común. Está pensado para leer los artefactos exportados por cada notebook de modelo y resumir resultados, residuos e interpretabilidad.

## Objetivos

- Consolidar métricas en una sola tabla.
- Comparar predicciones y residuos sobre el mismo conjunto de test.
- Revisar estabilidad territorial de los errores.
- Contrastar artefactos de interpretación de cada modelo.
- Dejar una conclusión ejecutiva sobre cuál conviene priorizar.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "output"
COMPARISON_DIR = OUTPUT_DIR / "20_model_comparison"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 200)
OUTPUT_DIR

## Registro de modelos

Ajustá esta tabla si más adelante cambiás nombres de carpetas o agregás variantes.

In [ ]:
MODEL_REGISTRY = {
    "lgwr": {
        "label": "LGWR / GWR",
        "artifact_dir": OUTPUT_DIR / "10_lgwr",
        "family": "spatial_local",
    },
    "sar": {
        "label": "SAR",
        "artifact_dir": OUTPUT_DIR / "11_sar",
        "family": "spatial_global",
    },
    "rf_kriging": {
        "label": "RF + Kriging",
        "artifact_dir": OUTPUT_DIR / "12_rf_kriging",
        "family": "hybrid",
    },
    "gnn": {
        "label": "GNN",
        "artifact_dir": OUTPUT_DIR / "13_gnn",
        "family": "graph",
    },
}
pd.DataFrame(MODEL_REGISTRY).T

## Helpers de carga

In [ ]:
def maybe_read_json(path):
    if path.exists():
        return json.loads(path.read_text())
    return None

def maybe_read_parquet(path):
    if path.exists():
        return pd.read_parquet(path)
    return None

def load_model_bundle(model_key, config):
    artifact_dir = config["artifact_dir"]
    bundle = {
        "model_key": model_key,
        "label": config["label"],
        "family": config["family"],
        "artifact_dir": artifact_dir,
        "metrics": maybe_read_json(artifact_dir / "metrics.json"),
        "run_config": maybe_read_json(artifact_dir / "run_config.json"),
        "predictions": maybe_read_parquet(artifact_dir / "test_predictions.parquet"),
        "residuals": maybe_read_parquet(artifact_dir / "residuals.parquet"),
        "interpretability": maybe_read_parquet(artifact_dir / "interpretability.parquet"),
    }
    bundle["available"] = any(value is not None for key, value in bundle.items() if key in {"metrics", "predictions", "residuals", "interpretability"})
    return bundle

model_bundles = {key: load_model_bundle(key, cfg) for key, cfg in MODEL_REGISTRY.items()}
availability = pd.DataFrame([
    {
        "model": bundle["label"],
        "available": bundle["available"],
        "has_metrics": bundle["metrics"] is not None,
        "has_predictions": bundle["predictions"] is not None,
        "has_residuals": bundle["residuals"] is not None,
        "has_interpretability": bundle["interpretability"] is not None,
    }
    for bundle in model_bundles.values()
])
availability

## Tabla de métricas

In [ ]:
metric_rows = []
for bundle in model_bundles.values():
    if bundle["metrics"] is None:
        continue
    row = {
        "model": bundle["label"],
        "family": bundle["family"],
        **bundle["metrics"],
    }
    metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows)
if not metrics_df.empty:
    sort_cols = [col for col in ["rmse", "mae"] if col in metrics_df.columns]
    metrics_df = metrics_df.sort_values(sort_cols) if sort_cols else metrics_df

metrics_df

## Visual rápido de performance

In [ ]:
if not metrics_df.empty:
    candidate_metrics = [col for col in ["rmse", "mae", "r2", "bias"] if col in metrics_df.columns]
    fig, axes = plt.subplots(1, len(candidate_metrics), figsize=(4 * len(candidate_metrics), 4))
    if len(candidate_metrics) == 1:
        axes = [axes]
    for ax, metric_name in zip(axes, candidate_metrics):
        chart_df = metrics_df.sort_values(metric_name, ascending=(metric_name != "r2"))
        sns.barplot(data=chart_df, x=metric_name, y="model", ax=ax, palette="deep")
        ax.set_title(metric_name.upper())
    plt.tight_layout()

## Consolidación de predicciones

In [ ]:
prediction_frames = []
for bundle in model_bundles.values():
    pred_df = bundle["predictions"]
    if pred_df is None:
        continue
    frame = pred_df.copy()
    frame["model"] = bundle["label"]
    prediction_frames.append(frame)

predictions_long = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
predictions_long.head() if not predictions_long.empty else predictions_long

## Residuos comparados

In [ ]:
if not predictions_long.empty and {"model", "residual"}.issubset(predictions_long.columns):
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=predictions_long, x="model", y="residual")
    plt.xticks(rotation=20)
    plt.title("Distribucion de residuos por modelo")
    plt.tight_layout()

## Error absoluto por zona o grupo

Si tus exports incluyen una columna territorial como `barrio`, `comuna` o similar, esta sección ayuda a detectar modelos que funcionan bien en promedio pero mal en áreas específicas.

In [ ]:
group_candidates = ["barrio", "comuna", "zone", "cluster"]
group_col = next((col for col in group_candidates if col in predictions_long.columns), None) if not predictions_long.empty else None

if group_col and {"model", "residual"}.issubset(predictions_long.columns):
    group_error = (
        predictions_long
        .assign(abs_error=lambda df: df["residual"].abs())
        .groupby(["model", group_col], as_index=False)["abs_error"]
        .mean()
        .sort_values(["model", "abs_error"], ascending=[True, False])
    )
    display(group_error.head(20))
else:
    print("No se detecto una columna territorial comun en los artefactos exportados.")

## Scatter real vs predicho

In [ ]:
if not predictions_long.empty and {"y_true", "y_pred", "model"}.issubset(predictions_long.columns):
    g = sns.FacetGrid(predictions_long, col="model", col_wrap=2, sharex=False, sharey=False, height=4)
    g.map_dataframe(sns.scatterplot, x="y_true", y="y_pred", alpha=0.35)
    g.set_axis_labels("y_true", "y_pred")
    g.figure.subplots_adjust(top=0.9)
    g.figure.suptitle("Real vs predicho por modelo")

## Comparación espacial rápida

Si los artifacts incluyen `lon` y `lat`, podés mapear residuos para ver patrones espaciales persistentes.

In [ ]:
if not predictions_long.empty and {"lon", "lat", "residual", "model"}.issubset(predictions_long.columns):
    g = sns.FacetGrid(predictions_long, col="model", col_wrap=2, sharex=False, sharey=False, height=4)
    g.map_dataframe(sns.scatterplot, x="lon", y="lat", hue="residual", palette="coolwarm", alpha=0.7)
    g.add_legend()
    g.figure.subplots_adjust(top=0.9)
    g.figure.suptitle("Mapa rapido de residuos por modelo")

## Artefactos de interpretación

Cada modelo exporta cosas distintas. La idea acá no es forzar una misma semántica, sino dejar una mesa de lectura rápida.

In [ ]:
interpretability_preview = {}
for bundle in model_bundles.values():
    interpret_df = bundle["interpretability"]
    if interpret_df is None:
        continue
    interpretability_preview[bundle["label"]] = interpret_df.head(10)

interpretability_preview

## Ranking tentativo

Podés ajustar los pesos si querés priorizar interpretabilidad, robustez espacial o error global.

In [ ]:
if not metrics_df.empty:
    ranking_df = metrics_df.copy()
    for col in ["rmse", "mae", "bias"]:
        if col in ranking_df.columns:
            ranking_df[f"rank_{col}"] = ranking_df[col].rank(method="min", ascending=True)
    if "r2" in ranking_df.columns:
        ranking_df["rank_r2"] = ranking_df["r2"].rank(method="min", ascending=False)
    rank_cols = [col for col in ranking_df.columns if col.startswith("rank_")]
    ranking_df["rank_score"] = ranking_df[rank_cols].mean(axis=1)
    ranking_df = ranking_df.sort_values("rank_score")
    ranking_df

## Export del comparador

In [ ]:
if not metrics_df.empty:
    metrics_df.to_csv(COMPARISON_DIR / "metrics_summary.csv", index=False)

if not predictions_long.empty:
    predictions_long.to_parquet(COMPARISON_DIR / "predictions_long.parquet", index=False)

comparison_meta = {
    "models_registered": list(MODEL_REGISTRY.keys()),
    "models_available": [key for key, bundle in model_bundles.items() if bundle["available"]],
}
(COMPARISON_DIR / "comparison_meta.json").write_text(json.dumps(comparison_meta, indent=2, ensure_ascii=False))

## Conclusiones

Completá este cierre cuando tengas los outputs:

- Mejor modelo global por RMSE/MAE:
- Mejor modelo en interpretabilidad:
- Mejor comportamiento espacial de residuos:
- Trade-off principal observado:
- Modelo recomendado para seguir iterando: